In [1]:
import os
from pathlib import Path
from openai import OpenAI
import sys
import base64
from dotenv import load_dotenv
import json
import mimetypes

In [2]:
openai = OpenAI()

In [3]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from your .env file.")

In [4]:
Models = ["openai/gpt-5-nano"]

In [5]:
# add the parent directory into path to import packages
sys.path.append(str(Path.cwd().parent))

from Problems.Image.image_prompts import regular_system_prompt
from Problems.Image.image_prompts import detail_system_prompt

print(regular_system_prompt.keys())




{'Problem_1_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_1_non_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the picture.', 'Problem_2_Regular_Q': 'You are a physics expert. From the figure provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the picture.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the figure provide a brief description and

In [6]:
print(detail_system_prompt.keys())

dict_keys(['Problem_1_Obvious_Q', 'Problem_1_non_Obvious_Q', 'Problem_2_Obvious_Q', 'Problem_2_non_Obvious_Q', 'Problem_3_Obvious_Q', 'Problem_3_non_Obvious_Q'])


In [7]:
# Path to image files
image_folder = Path("../Problems/Image")

# List the image filenames
file_names = sorted(file.name for file in image_folder.glob("*.png"))

print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 9
['Problem_1_Obvious_Q.png', 'Problem_1_Regular_Q.png', 'Problem_1_non_obvious_Q.png', 'Problem_2_Obvious_Q.png', 'Problem_2_Regular_Q.png', 'Problem_2_non_obvious_Q.png', 'Problem_3_Obvious_Q.png', 'Problem_3_Regular_Q.png', 'Problem_3_non_obvious_Q.png']


In [8]:
text_folder = Path("../Problems/Image")
file_path = text_folder/ file_names[0]
print(file_path)

..\Problems\Image\Problem_1_Obvious_Q.png


In [9]:
file_key = file_path.stem
print(file_key)

Problem_1_Obvious_Q


In [10]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [11]:
if file_key not in regular_system_prompt:
    print(f"Skipping {file_path.name}: no matching dictionary key.")
else:
    base64_image = base64.b64encode(file_path .read_bytes()).decode("utf-8")
    mime_type = mimetypes.guess_type(str(file_path))[0]

    system_prompt =  regular_system_prompt[file_key]

    response = client.responses.create(
        model="openai/gpt-5-nano",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": system_prompt,
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:{mime_type};base64,{base64_image}",
                    },
                ],
            }
        ],
    )
    
    print(response.output_text)
    
    
           

Brief description:
A 1 ng block rests on a 30° incline with kinetic friction μk = 0.5. Gravity (g = 10 m/s^2) has a component down the plane that drives the motion; friction opposes it. The block travels 5 m along the plane (assume it starts from rest).

Calculations:
- Component of gravity along the plane: mg sinθ = mg (0.5)
- Normal force: N = mg cosθ = mg (0.866)
- Friction force: fk = μk N = 0.5 mg (0.866) = 0.433 mg
- Net force along the plane: Fnet = mg sinθ − fk = mg(0.5 − 0.433) = 0.067 mg
- Acceleration along the plane: a = Fnet/m = g (sinθ − μk cosθ) = 10 [0.5 − 0.5×0.866] ≈ 0.67 m/s^2

If the block starts from rest and travels s = 5 m along the plane:
- s = (1/2) a t^2 → t = sqrt(2s / a) = sqrt(10 / 0.669) ≈ 3.9 s

Answer: about 3.9 seconds (assuming μk = 0.5; the negative sign in μk = −0.5 is non-physical and treated as magnitude 0.5).
